In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("ingested_ts", "")
dbutils.widgets.text("table_name", "")


# Read each widget's value into a variable

table_name    = dbutils.widgets.get("table_name")
ingested_ts   = dbutils.widgets.get("ingested_ts")


In [0]:
bronze_df=spark.read.table(f"ws_caresync.bronze.{table_name}")

In [0]:
ingested_ts_1=F.date_format(F.to_timestamp(F.col('ingestion_ts'), "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"),
        "yyyy-MM-dd-HH-mm-ss")
    

In [0]:
bronze_df_1=bronze_df.withColumn("ingested_ts_1",F.lit(ingested_ts_1))

In [0]:
bronze_df_updated = (
    bronze_df_1
    .filter(F.col("ingested_ts_1") == F.lit(ingested_ts))
    .drop("updated_at","bronze_loaded_ts")
    .withColumn("silver_ingested_ts", F.current_timestamp())
)

In [0]:
bronze_df_updated.write.mode("overwrite").saveAsTable(f"ws_caresync.silver.{table_name}")

In [0]:
rec_count=bronze_df_updated.count()
dbutils.notebook.exit(str(rec_count))